# 第 5 章互动实验：角点与旋转重复性

本实验在合成标记上调用 OpenCV Harris 基线，仅观察阈值和旋转效果；不会给出 `5-1_Harris响应.py` 的结构张量实现。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

base = np.zeros((180, 220), dtype=np.uint8)
for y in range(30, 151, 30):
    for x in range(35, 186, 30):
        if (x // 30 + y // 30) % 2: cv2.rectangle(base, (x, y), (x+20, y+20), 220, -1)
cv2.rectangle(base, (12, 12), (207, 167), 130, 2)
center = (base.shape[1] / 2, base.shape[0] / 2)

In [ ]:
def detect(gray, quality):
    pts = cv2.goodFeaturesToTrack(gray, maxCorners=200, qualityLevel=quality, minDistance=7, blockSize=5, useHarrisDetector=True, k=0.04)
    return np.empty((0, 2), np.float32) if pts is None else pts.reshape(-1, 2)

def show_corners(angle=20, quality=0.02):
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(base, M, (base.shape[1], base.shape[0]))
    points_a, points_b = detect(base, quality), detect(rotated, quality)
    views = [cv2.cvtColor(base, cv2.COLOR_GRAY2RGB), cv2.cvtColor(rotated, cv2.COLOR_GRAY2RGB)]
    for view, points in zip(views, [points_a, points_b]):
        for x, y in points: cv2.circle(view, (round(float(x)), round(float(y))), 3, (255, 60, 30), -1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, view, title in zip(axes, views, [f'原图：{len(points_a)} 点', f'旋转 {angle}°：{len(points_b)} 点']):
        ax.imshow(view); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_corners,
    angle=IntSlider(min=0, max=60, step=5, value=20),
    quality=FloatSlider(min=0.005, max=0.15, step=0.005, value=0.02, readout_format='.3f'),
);

## 破坏实验与记录

- 提高 `qualityLevel`，记录检测点数量何时骤减。
- 比较 0°、30°、45°、60°，不要只比较点数，还要观察位置是否对应。
- 对 `base` 先做强高斯模糊，观察定位和低对比度点。

在 `实验记录.md` 中区分“检测数量”和“可重复性”。